# Part 1. fluency2 error analysis (WMT25 ESA + WMT22-24 MQM)
Comprehensive error analysis notebook for **fluency2** on:
- WMT25 ESA (`merged_all_metrics.parquet`)
- WMT22-24 MQM (`all_metrics_mqm.parquet`)


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import spearmanr, kendalltau
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Optional viz lib (graceful fallback if not installed)
try:
    import seaborn as sns
    SEABORN_AVAILABLE = True
except ImportError:
    sns = None
    SEABORN_AVAILABLE = False
    print("seaborn not installed - heatmaps will use matplotlib fallback")

# Optional NLP libs (graceful fallback if not installed)
try:
    import spacy
    SPACY_AVAILABLE = True
except ImportError:
    SPACY_AVAILABLE = False
    print("spacy not installed - morphological analysis will be skipped")

try:
    from sacrebleu.metrics import CHRF, BLEU
    SACREBLEU_AVAILABLE = True
except ImportError:
    SACREBLEU_AVAILABLE = False

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
WMT25_PATH = ROOT / 'data/wmt25/merged_all_metrics.parquet'
MQM_PATH = ROOT / 'data/wmt_mqm/all_metrics_mqm.parquet'

COLORS = {
    'fluency2': '#2D4A8A',
    'gemba':    '#8A4A2D',
    'metricx':  '#2D8A5A',
    'chrf':     '#8A2D5A',
    'xcomet':   '#5A2D8A',
    'comet':    '#8A7A2D',
}

SYNTACTIC_DISTANCES = {
    'EN→uk_UA': 0.153, 'EN→is_IS': 0.154, 'EN→ru_RU': 0.183,
    'EN→sr_Cyrl_RS': 0.185, 'EN→et_EE': 0.195, 'EN→zh_CN': 0.268,
    'EN→cs_CZ': 0.335, 'EN→bho_IN': 0.337, 'EN→ar_EG': 0.366,
    'EN→mas_KE': 0.431, 'EN→ja_JP': 0.456, 'EN→it_IT': 0.115,
}

# Subdimension palette + display order ("sort by color")
SUBDIM_COLORS = {
    'idiomatic': '#2D4A8A',     # blue
    'collocational': '#2D8A5A', # green
    'discourse': '#5A2D8A',     # purple
    'pragmatic': '#8A4A2D',     # brown
    'calque': '#8A2D5A',        # magenta
}
SUBDIM_ORDER = list(SUBDIM_COLORS.keys())

# Language typology metadata
LANG_META = {
    'EN→uk_UA': {'script': 'cyrillic', 'morphology': 'fusional_rich',
                 'word_order': 'free', 'family': 'slavic'},
    'EN→is_IS': {'script': 'latin', 'morphology': 'fusional_rich',
                 'word_order': 'v2', 'family': 'germanic'},
    'EN→ru_RU': {'script': 'cyrillic', 'morphology': 'fusional_rich',
                 'word_order': 'free', 'family': 'slavic'},
    'EN→sr_Cyrl_RS': {'script': 'cyrillic', 'morphology': 'fusional_rich',
                       'word_order': 'free', 'family': 'slavic'},
    'EN→et_EE': {'script': 'latin', 'morphology': 'agglutinative',
                  'word_order': 'sov_flexible', 'family': 'uralic'},
    'EN→zh_CN': {'script': 'cjk', 'morphology': 'isolating',
                  'word_order': 'svo', 'family': 'sino_tibetan'},
    'EN→cs_CZ': {'script': 'latin', 'morphology': 'fusional_rich',
                  'word_order': 'free', 'family': 'slavic'},
    'EN→bho_IN': {'script': 'devanagari', 'morphology': 'fusional_moderate',
                   'word_order': 'sov', 'family': 'indic'},
    'EN→ar_EG': {'script': 'arabic', 'morphology': 'root_pattern',
                  'word_order': 'vso_flexible', 'family': 'semitic'},
    'EN→mas_KE': {'script': 'latin', 'morphology': 'agglutinative',
                   'word_order': 'vso', 'family': 'nilo_saharan'},
    'EN→ja_JP': {'script': 'cjk_mixed', 'morphology': 'agglutinative',
                  'word_order': 'sov', 'family': 'japonic'},
    'EN→it_IT': {'script': 'latin', 'morphology': 'fusional_moderate',
                  'word_order': 'svo_flexible', 'family': 'romance'},
}

print("Setup complete")
print(f"WMT25 exists: {WMT25_PATH.exists()}")
print(f"MQM exists: {MQM_PATH.exists()}")


In [ ]:
df25 = pd.read_parquet(WMT25_PATH)
dfmqm = pd.read_parquet(MQM_PATH)

# Add typology metadata to WMT25
for key, meta in LANG_META.items():
    for col, val in meta.items():
        df25.loc[df25['pair'] == key, col] = val

# Add syntactic distance
df25['syntactic_distance'] = df25['pair'].map(SYNTACTIC_DISTANCES)

# Fair comparison subsets
WMT25_METRICS = ['fluency2', 'gemba_score', 'metricx_score',
                  'xcomet_score', 'cometKiwi_score', 'chrf_pp']
mask25 = df25['human_score'].notna()
for m in WMT25_METRICS:
    if m in df25.columns:
        mask25 = mask25 & df25[m].notna()
df25_fair = df25[mask25].copy()

MQM_METRICS = [c for c in ['fluency2','gemba_mqm','gemba_da','chrf_pp']
               if c in dfmqm.columns]
mask_mqm = dfmqm['human_mqm_score'].notna()
for m in MQM_METRICS:
    mask_mqm = mask_mqm & dfmqm[m].notna()
dfmqm_fair = dfmqm[mask_mqm].copy()

# Segment length
df25_fair['src_len'] = df25_fair['src_text'].fillna('').apply(
    lambda x: len(str(x).split()))
df25_fair['mt_len'] = df25_fair['mt_text'].fillna('').apply(
    lambda x: len(str(x).split()))
df25_fair['len_ratio'] = df25_fair['mt_len'] / (df25_fair['src_len'] + 1)

def len_bucket(n):
    if n <= 5:   return '1-5'
    if n <= 15:  return '6-15'
    if n <= 30:  return '16-30'
    return '31+'

df25_fair['len_bucket'] = df25_fair['src_len'].apply(len_bucket)

print(f"WMT25 fair: {len(df25_fair)} rows")
print(f"MQM fair: {len(dfmqm_fair)} rows")
print(f"WMT25 pairs: {df25_fair['pair'].nunique()}")


## 1. Quality quintile analysis
Does fluency2 discriminate better on bad translations (easy) vs good ones (hard)?
The 'top-quintile collapse' is a universal failure mode of all MT metrics.

In [ ]:
df25_fair['human_quintile'] = pd.qcut(
    df25_fair['human_score'], 5,
    labels=['Q1\n(worst)', 'Q2', 'Q3', 'Q4', 'Q5\n(best)'])

quintile_results = []
for q in df25_fair['human_quintile'].cat.categories:
    sub = df25_fair[df25_fair['human_quintile'] == q]
    for metric in WMT25_METRICS:
        if metric not in sub.columns:
            continue
        s = sub[sub[metric].notna()]
        if len(s) < 10:
            continue
        tau, _ = kendalltau(s[metric], s['human_score'])
        quintile_results.append({
            'quintile': q, 'metric': metric,
            'tau': round(tau, 3), 'n': len(s)
        })

df_quint = pd.DataFrame(quintile_results)

fig, ax = plt.subplots(figsize=(12, 5))
for metric in WMT25_METRICS:
    sub = df_quint[df_quint['metric'] == metric]
    if sub.empty:
        continue
    color = COLORS.get(metric.replace('_score','').replace('_pp',''), '#888')
    label = metric.replace('_score','').replace('_pp','')
    ax.plot(sub['quintile'].astype(str), sub['tau'],
            marker='o', color=color, label=label, linewidth=2)

ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Human Score Quintile')
ax.set_ylabel('Kendall τ')
ax.set_title('Quality Quintile Analysis: Does fluency2 work on top-quality translations?')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.savefig('/tmp/fig_quintile.png', dpi=150)
plt.show()

print("Key finding: top-quintile collapse?")
q5_f2 = df_quint[(df_quint['metric']=='fluency2') &
                  (df_quint['quintile'].astype(str).str.contains('Q5'))]
print(f"fluency2 Q5 τ: {q5_f2['tau'].values}")


### 2. Length analysis

In [ ]:
len_results = []
for bucket in ['1-5', '6-15', '16-30', '31+']:
    sub = df25_fair[df25_fair['len_bucket'] == bucket]
    for metric in WMT25_METRICS:
        if metric not in sub.columns:
            continue
        s = sub[sub[metric].notna()]
        if len(s) < 5:
            continue
        rho, _ = spearmanr(s[metric], s['human_score'])
        mae = np.mean(np.abs(
            (s[metric] - s[metric].mean()) / s[metric].std() -
            (s['human_score'] - s['human_score'].mean()) / s['human_score'].std()
        ))
        len_results.append({
            'bucket': bucket, 'metric': metric,
            'spearman': round(rho, 3), 'mae_z': round(mae, 3),
            'n': len(s)
        })

df_len = pd.DataFrame(len_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Spearman by length
ax = axes[0]
for metric in WMT25_METRICS:
    sub = df_len[df_len['metric'] == metric]
    if sub.empty: continue
    color = COLORS.get(metric.replace('_score','').replace('_pp',''), '#888')
    ax.plot(sub['bucket'], sub['spearman'],
            marker='o', color=color,
            label=metric.replace('_score','').replace('_pp',''),
            linewidth=2)
ax.set_xlabel('Source Length (tokens)')
ax.set_ylabel('Spearman ρ')
ax.set_title('Correlation by Segment Length')
ax.legend(fontsize=8)

# N by bucket
ax = axes[1]
n_by_bucket = df25_fair.groupby('len_bucket').size()
ax.bar(n_by_bucket.index, n_by_bucket.values, color=COLORS['fluency2'], alpha=0.8)
ax.set_xlabel('Length Bucket')
ax.set_ylabel('N segments')
ax.set_title('Segment Count by Length')

plt.tight_layout()
plt.savefig('/tmp/fig_length.png', dpi=150)
plt.show()

print("\nCorrelation by length bucket:")
pivot_len = df_len.pivot_table(
    index='bucket', columns='metric', values='spearman'
).round(3)
print(pivot_len.to_string())


### 3. Typological clustering
Group language pairs by morphological type, script, word order.
Does fluency2 performance cluster by typology?


In [ ]:
# Add typology to per-LP correlation
lp_stats = []
for lp, group in df25_fair.groupby('pair'):
    meta = LANG_META.get(lp, {})
    dist = SYNTACTIC_DISTANCES.get(lp, np.nan)
    for metric in WMT25_METRICS:
        if metric not in group.columns:
            continue
        sub = group[group[metric].notna() & group['human_score'].notna()]
        if len(sub) < 5:
            continue
        rho, _ = spearmanr(sub[metric], sub['human_score'])
        tau, _ = kendalltau(sub[metric], sub['human_score'])

        # Pairwise accuracy
        pred = sub[metric].values
        human = sub['human_score'].values
        correct = total = 0
        for i in range(len(pred)):
            for j in range(i+1, len(pred)):
                if human[i] == human[j]:
                    continue
                total += 1
                if (pred[i] > pred[j]) == (human[i] > human[j]):
                    correct += 1
        pa = correct / total if total > 0 else np.nan

        lp_stats.append({
            'pair': lp, 'metric': metric,
            'spearman': round(rho, 3), 'tau': round(tau, 3),
            'pa': round(pa, 3), 'n': len(sub),
            'syntactic_dist': dist,
            'morphology': meta.get('morphology', 'unknown'),
            'script': meta.get('script', 'unknown'),
            'word_order': meta.get('word_order', 'unknown'),
            'family': meta.get('family', 'unknown'),
        })

df_lp = pd.DataFrame(lp_stats)

## 3.2 Heatmap: PA by language pair × metric
fig, ax = plt.subplots(figsize=(12, 6))
pivot_pa = df_lp.pivot_table(
    index='pair', columns='metric', values='pa'
)

# Sort language pairs by fluency2 PA (worst→best), if available
if 'fluency2' in pivot_pa.columns:
    pivot_pa = pivot_pa.sort_values('fluency2', ascending=True)

# Sort metric columns by palette order ("by color")
metric_order = [
    'fluency2',
    'gemba_score',
    'metricx_score',
    'xcomet_score',
    'cometKiwi_score',
    'chrf_pp',
]
metric_order = [m for m in metric_order if m in pivot_pa.columns]
# Keep any unexpected metrics at the end
remaining = [m for m in pivot_pa.columns if m not in metric_order]
pivot_pa = pivot_pa[metric_order + remaining]

if SEABORN_AVAILABLE:
    sns.heatmap(pivot_pa, annot=True, fmt='.3f', cmap='RdYlGn',
                center=0.65, ax=ax, linewidths=0.5)
else:
    im = ax.imshow(pivot_pa.values, aspect='auto', cmap='RdYlGn', vmin=0.5, vmax=0.8)
    ax.set_xticks(range(pivot_pa.shape[1]))
    ax.set_xticklabels(pivot_pa.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(pivot_pa.shape[0]))
    ax.set_yticklabels(pivot_pa.index, fontsize=8)
    for i in range(pivot_pa.shape[0]):
        for j in range(pivot_pa.shape[1]):
            v = pivot_pa.values[i, j]
            if pd.notna(v):
                ax.text(j, i, f"{v:.3f}", ha='center', va='center', fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

ax.set_title('Pairwise Accuracy by Language Pair × Metric')
plt.tight_layout()
plt.savefig('/tmp/fig_lp_heatmap.png', dpi=150)
plt.show()

## 3.3 Group by morphological type
print("\n=== FLUENCY2 PA BY MORPHOLOGICAL TYPE ===")
morph_stats = df_lp[df_lp['metric']=='fluency2'].groupby('morphology').agg(
    mean_pa=('pa', 'mean'),
    mean_spearman=('spearman', 'mean'),
    n_pairs=('pair', 'count')
).round(3)
print(morph_stats.to_string())

print("\n=== FLUENCY2 PA BY SCRIPT ===")
script_stats = df_lp[df_lp['metric']=='fluency2'].groupby('script').agg(
    mean_pa=('pa', 'mean'),
    mean_spearman=('spearman', 'mean'),
    n_pairs=('pair', 'count')
).round(3)
print(script_stats.to_string())

print("\n=== FLUENCY2 PA BY WORD ORDER ===")
wo_stats = df_lp[df_lp['metric']=='fluency2'].groupby('word_order').agg(
    mean_pa=('pa', 'mean'),
    mean_spearman=('spearman', 'mean'),
    n_pairs=('pair', 'count')
).round(3)
print(wo_stats.to_string())

## 3.4 Fluency2 advantage by typology vs each baseline
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax_i, (group_col, title) in enumerate([
    ('morphology', 'Morphological Type'),
    ('script', 'Writing Script'),
    ('word_order', 'Word Order'),
]):
    ax = axes[ax_i]
    f2 = df_lp[df_lp['metric']=='fluency2'][
        ['pair', group_col, 'pa']].rename(columns={'pa': 'f2_pa'})
    chrf = df_lp[df_lp['metric']=='chrf_pp'][
        ['pair', 'pa']].rename(columns={'pa': 'chrf_pa'})
    merged = f2.merge(chrf, on='pair')
    merged['advantage'] = merged['f2_pa'] - merged['chrf_pa']

    order = merged.groupby(group_col)['advantage'].mean().sort_values().index
    colors_bar = [COLORS['fluency2'] if v >= 0 else COLORS['chrf']
                  for v in merged.groupby(group_col)['advantage'].mean().loc[order]]
    ax.barh(range(len(order)),
            merged.groupby(group_col)['advantage'].mean().loc[order],
            color=colors_bar, alpha=0.8)
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels(order, fontsize=9)
    ax.axvline(0, color='gray', linestyle='--')
    ax.set_xlabel('fluency2 PA − chrF++ PA')
    ax.set_title(f'Advantage by {title}')

plt.suptitle('Where does fluency2 beat chrF++ by typological group?',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/fig_typology_advantage.png', dpi=150)
plt.show()


### 4. Syntactic distance


In [ ]:
## 4.1 Correlation vs syntactic distance scatter — all metrics
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
baselines = ['gemba_score', 'metricx_score', 'xcomet_score',
             'cometKiwi_score', 'chrf_pp']

for ax_i, baseline in enumerate(baselines):
    ax = axes[ax_i // 3][ax_i % 3]

    f2_pa = df_lp[df_lp['metric']=='fluency2'][['pair','pa','syntactic_dist']]
    bl_pa = df_lp[df_lp['metric']==baseline][['pair','pa']].rename(
        columns={'pa':'bl_pa'})
    merged = f2_pa.merge(bl_pa, on='pair').dropna()
    merged['advantage'] = merged['pa'] - merged['bl_pa']

    ax.scatter(merged['syntactic_dist'], merged['advantage'],
               color=COLORS['fluency2'], s=80, zorder=5)
    if len(merged) > 2:
        m, b = np.polyfit(merged['syntactic_dist'], merged['advantage'], 1)
        x_line = np.linspace(merged['syntactic_dist'].min(),
                              merged['syntactic_dist'].max(), 100)
        ax.plot(x_line, m*x_line + b, color=COLORS.get(
            baseline.replace('_score','').replace('_pp',''), '#888'),
                linewidth=2, alpha=0.7)
        rho, p = spearmanr(merged['syntactic_dist'], merged['advantage'])
        ax.set_title(f'vs {baseline.replace("_score","").replace("_pp","")}\n'
                     f'ρ={rho:.3f}, p={p:.3f}', fontsize=9)

    for _, row in merged.iterrows():
        ax.annotate(row['pair'].replace('EN→',''),
                    (row['syntactic_dist'], row['advantage']),
                    fontsize=7, xytext=(3,3), textcoords='offset points')
    ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('Syntactic Distance')
    ax.set_ylabel('fluency2 PA − baseline PA')

# Last subplot: all baselines overlay
ax = axes[1][2]
for baseline in baselines:
    f2_pa = df_lp[df_lp['metric']=='fluency2'][['pair','pa','syntactic_dist']]
    bl_pa = df_lp[df_lp['metric']==baseline][['pair','pa']].rename(
        columns={'pa':'bl_pa'})
    merged = f2_pa.merge(bl_pa, on='pair').dropna()
    merged['advantage'] = merged['pa'] - merged['bl_pa']
    if len(merged) > 2:
        rho, p = spearmanr(merged['syntactic_dist'], merged['advantage'])
        color = COLORS.get(
            baseline.replace('_score','').replace('_pp',''), '#888')
        ax.scatter([rho], [p], color=color, s=100,
                   label=baseline.replace('_score','').replace('_pp',''), zorder=5)

ax.axhline(0.05, color='red', linestyle='--', alpha=0.7, label='p=0.05')
ax.set_xlabel('Spearman ρ (distance vs advantage)')
ax.set_ylabel('p-value')
ax.set_title('Typological signal: all baselines\n(bottom-left = strongest signal)')
ax.legend(fontsize=8)
ax.invert_yaxis()

plt.suptitle('fluency2 Advantage vs Syntactic Distance — All Baselines',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/fig_distance_all.png', dpi=150)
plt.show()


### 5. Issue type and subdimension analysis


In [ ]:
## 5.1 issue_type distribution by language pair
if 'issue_type' in df25_fair.columns:
    print("=== ISSUE TYPE DISTRIBUTION BY PAIR ===")
    issue_by_pair = df25_fair.groupby(['pair', 'issue_type']).size().unstack(
        fill_value=0)
    issue_pct = issue_by_pair.div(issue_by_pair.sum(axis=1), axis=0).round(3)
    print(issue_pct.to_string())

    fig, ax = plt.subplots(figsize=(12, 6))
    issue_pct.drop(columns=['none'], errors='ignore').plot(
        kind='bar', ax=ax, colormap='tab10')
    ax.set_xlabel('Language Pair')
    ax.set_ylabel('Fraction of segments')
    ax.set_title('Issue Type Distribution by Language Pair')
    ax.legend(loc='upper right', fontsize=8)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('/tmp/fig_issue_type.png', dpi=150)
    plt.show()

## 5.2 Subdimension correlation heatmap by pair
subdims = [c for c in ['idiomatic','collocational','discourse',
                        'pragmatic','calque']
           if c in df25_fair.columns]

if subdims:
    subdim_lp = []
    for lp, group in df25_fair.groupby('pair'):
        for dim in subdims:
            sub = group[group[dim].notna() & group['human_score'].notna()]
            if len(sub) < 5: continue
            rho, _ = spearmanr(sub[dim], sub['human_score'])
            subdim_lp.append({'pair': lp, 'subdim': dim,
                               'spearman': round(rho, 3)})

    df_subdim_lp = pd.DataFrame(subdim_lp)
    pivot_subdim = df_subdim_lp.pivot_table(
        index='pair', columns='subdim', values='spearman')

    # Sort columns by "color order" (preferred display order)
    col_order = [c for c in SUBDIM_ORDER if c in pivot_subdim.columns]
    remaining = [c for c in pivot_subdim.columns if c not in col_order]
    pivot_subdim = pivot_subdim[col_order + remaining]

    fig, ax = plt.subplots(figsize=(10, 7))
    if SEABORN_AVAILABLE:
        sns.heatmap(pivot_subdim, annot=True, fmt='.2f', cmap='RdYlGn',
                    center=0.4, ax=ax, linewidths=0.5)
        # Color x tick labels by subdimension palette
        for tick in ax.get_xticklabels():
            name = tick.get_text()
            tick.set_color(SUBDIM_COLORS.get(name, 'black'))
    else:
        im = ax.imshow(pivot_subdim.values, aspect='auto', cmap='RdYlGn', vmin=0.0, vmax=1.0)
        ax.set_xticks(range(pivot_subdim.shape[1]))
        ax.set_xticklabels(pivot_subdim.columns, rotation=45, ha='right', fontsize=8)
        for tick in ax.get_xticklabels():
            name = tick.get_text()
            tick.set_color(SUBDIM_COLORS.get(name, 'black'))
        ax.set_yticks(range(pivot_subdim.shape[0]))
        ax.set_yticklabels(pivot_subdim.index, fontsize=8)
        for i in range(pivot_subdim.shape[0]):
            for j in range(pivot_subdim.shape[1]):
                v = pivot_subdim.values[i, j]
                if pd.notna(v):
                    ax.text(j, i, f"{v:.2f}", ha='center', va='center', fontsize=7)
        fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

    ax.set_title('Subdimension Correlations with Human Score by Language Pair')
    plt.tight_layout()
    plt.savefig('/tmp/fig_subdim_lp.png', dpi=150)
    plt.show()

## 5.3 low_confidence analysis
if 'low_confidence' in df25_fair.columns:
    print("=== LOW_CONFIDENCE ANALYSIS ===")

    # Does low_confidence flag predict larger errors?
    df25_fair['metric_human_diff'] = abs(
        (df25_fair['fluency2'] - df25_fair['fluency2'].mean()) /
        df25_fair['fluency2'].std() -
        (df25_fair['human_score'] - df25_fair['human_score'].mean()) /
        df25_fair['human_score'].std()
    )

    lc_stats = df25_fair.groupby('low_confidence')['metric_human_diff'].agg(
        ['mean', 'median', 'std', 'count']).round(3)
    print(lc_stats.to_string())
    print("\n→ If low_confidence=True has higher mean error, the flag is informative")

    # phrase_verified
    if 'phrase_verified' in df25_fair.columns:
        pv_stats = df25_fair.groupby('phrase_verified')['metric_human_diff'].agg(
            ['mean', 'count']).round(3)
        print("\n=== PHRASE_VERIFIED vs ERROR ===")
        print(pv_stats.to_string())


### 6. Top-20 failure analysis


In [ ]:
# Compute normalized disagreement
df25_fair['f2_z'] = (
    (df25_fair['fluency2'] - df25_fair['fluency2'].mean()) /
    df25_fair['fluency2'].std()
)
df25_fair['human_z'] = (
    (df25_fair['human_score'] - df25_fair['human_score'].mean()) /
    df25_fair['human_score'].std()
)
df25_fair['disagreement'] = abs(df25_fair['f2_z'] - df25_fair['human_z'])

# Also flag direction of error
df25_fair['f2_overestimates'] = df25_fair['f2_z'] > df25_fair['human_z']

print("=== OVERESTIMATE vs UNDERESTIMATE ===")
print(df25_fair['f2_overestimates'].value_counts())
print(f"\nfluency2 overestimates (scores too high): "
      f"{df25_fair['f2_overestimates'].mean():.1%} of segments")

# Top 20 worst
worst20 = df25_fair.nlargest(20, 'disagreement')[[
    'pair', 'src_text', 'mt_text', 'human_score', 'fluency2',
    'gemba_score', 'disagreement', 'f2_overestimates',
    'issue_type', 'severity', 'low_confidence', 'main_issue',
    'idiomatic', 'collocational', 'discourse', 'pragmatic', 'calque'
]]

print("\n=== TOP 20 WORST FLUENCY2 DISAGREEMENTS ===")
for i, (_, row) in enumerate(worst20.iterrows()):
    direction = "OVER" if row['f2_overestimates'] else "UNDER"
    print(f"\n[{i+1}] {row['pair']} | {direction}ESTIMATE | "
          f"disagreement={row['disagreement']:.2f}")
    print(f"  Human: {row['human_score']:.1f} | "
          f"fluency2: {row['fluency2']:.2f} | "
          f"GEMBA: {row.get('gemba_score', 'N/A')}")
    print(f"  Issue: {row['issue_type']} | "
          f"Severity: {row['severity']} | "
          f"LowConf: {row['low_confidence']}")
    print(f"  Subdims: id={row.get('idiomatic','?'):.1f} "
          f"col={row.get('collocational','?'):.1f} "
          f"dis={row.get('discourse','?'):.1f} "
          f"prag={row.get('pragmatic','?'):.1f} "
          f"cal={row.get('calque','?'):.1f}")
    print(f"  Main issue: {str(row.get('main_issue',''))[:100]}")
    print(f"  SRC: {str(row.get('src_text',''))[:120]}")
    print(f"  MT:  {str(row.get('mt_text',''))[:120]}")

# Distribution of issue types in top-20
print("\n=== ISSUE TYPES IN TOP-20 FAILURES ===")
print(worst20['issue_type'].value_counts())
print("\n=== OVERESTIMATE RATE BY ISSUE TYPE ===")
print(df25_fair.groupby('issue_type')['f2_overestimates'].mean().sort_values(
    ascending=False).round(3))


### 7. MQM Error analysis


In [ ]:
## 7.1 fluency2 error on MQM by year+pair
dfmqm_fair['f2_z'] = (
    (dfmqm_fair['fluency2'] - dfmqm_fair['fluency2'].mean()) /
    dfmqm_fair['fluency2'].std()
)
dfmqm_fair['human_z'] = (
    (dfmqm_fair['human_mqm_score'] - dfmqm_fair['human_mqm_score'].mean()) /
    dfmqm_fair['human_mqm_score'].std()
)
dfmqm_fair['disagreement'] = abs(dfmqm_fair['f2_z'] - dfmqm_fair['human_z'])

print("=== MQM: DISAGREEMENT BY YEAR+PAIR ===")
print(dfmqm_fair.groupby(['year','pair'])['disagreement'].agg(
    ['mean','median','count']).round(3).to_string())

## 7.2 Quality quintile on MQM
# pd.qcut(..., duplicates='drop') can reduce the number of bins when many ties exist.
# To avoid label/bin-edge mismatches, compute bin codes first, then map to as-many labels as exist.

mqm_bin = pd.qcut(
    dfmqm_fair['human_mqm_score'], 5,
    labels=False,
    duplicates='drop'
)

n_bins = int(mqm_bin.nunique(dropna=True))
labels = ['Q1\n(worst)', 'Q2', 'Q3', 'Q4', 'Q5\n(best)'][:n_bins]

dfmqm_fair['mqm_quintile'] = mqm_bin.map(
    {i: labels[i] for i in range(n_bins)}
).astype('category')

mqm_quint = []
for q in dfmqm_fair['mqm_quintile'].cat.categories:
    sub = dfmqm_fair[dfmqm_fair['mqm_quintile'] == q]
    for metric in MQM_METRICS:
        s = sub[sub[metric].notna()]
        if len(s) < 5: continue
        tau, _ = kendalltau(s[metric], s['human_mqm_score'])
        mqm_quint.append({'quintile': q, 'metric': metric,
                           'tau': round(tau, 3), 'n': len(s)})

df_mqm_q = pd.DataFrame(mqm_quint)

fig, ax = plt.subplots(figsize=(10, 5))
for metric in MQM_METRICS:
    sub = df_mqm_q[df_mqm_q['metric'] == metric]
    if sub.empty: continue
    color = COLORS.get(metric.replace('_mqm','').replace('_da','').replace('_pp',''),
                        '#888')
    ax.plot(sub['quintile'].astype(str), sub['tau'],
            marker='o', label=metric, color=color, linewidth=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Human MQM Quintile')
ax.set_ylabel('Kendall τ')
ax.set_title('MQM Quality Quintile Analysis')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/fig_mqm_quintile.png', dpi=150)
plt.show()

## 7.3 Top-20 worst on MQM
worst20_mqm = dfmqm_fair.nlargest(20, 'disagreement')[[
    'year', 'pair', 'original_text', 'translation',
    'human_mqm_score', 'fluency2', 'gemba_mqm',
    'disagreement', 'issue_type', 'severity', 'low_confidence', 'main_issue'
]]

print("\n=== TOP 20 WORST fluency2 DISAGREEMENTS ON MQM ===")
for i, (_, row) in enumerate(worst20_mqm.iterrows()):
    print(f"\n[{i+1}] {row['year']} {row['pair']} | "
          f"disagreement={row['disagreement']:.2f}")
    print(f"  Human MQM: {row['human_mqm_score']:.1f} | "
          f"fluency2: {row['fluency2']:.2f} | "
          f"GEMBA-MQM: {row.get('gemba_mqm', 'N/A')}")
    print(f"  Issue: {row['issue_type']} | Severity: {row['severity']}")
    print(f"  Main issue: {str(row.get('main_issue',''))[:100]}")
    print(f"  SRC: {str(row.get('original_text',''))[:120]}")
    print(f"  MT:  {str(row.get('translation',''))[:120]}")


### 8. Calibration analysis


In [ ]:
## 8. Reliability diagram + score distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# WMT25: reliability diagram
ax = axes[0]
df25_fair['f2_decile'] = pd.qcut(df25_fair['fluency2'], 10,
                                   labels=False, duplicates='drop')
cal_data = df25_fair.groupby('f2_decile').agg(
    mean_f2=('fluency2', 'mean'),
    mean_human=('human_score', 'mean'),
    n=('fluency2', 'count')
).reset_index()
ax.scatter(cal_data['mean_f2'], cal_data['mean_human'],
           s=cal_data['n']/5, color=COLORS['fluency2'], alpha=0.8)
min_v = min(cal_data['mean_f2'].min(), cal_data['mean_human'].min())
max_v = max(cal_data['mean_f2'].max(), cal_data['mean_human'].max())
ax.plot([min_v, max_v], [min_v, max_v], 'r--', alpha=0.5, label='Perfect')
ax.set_xlabel('Mean fluency2 score (decile)')
ax.set_ylabel('Mean human score (decile)')
ax.set_title('Calibration: WMT25 ESA\n(bubble size = n)')
ax.legend()

# Score distributions overlay
ax = axes[1]
ax.hist(df25_fair['fluency2'], bins=30, alpha=0.6,
        color=COLORS['fluency2'], label='fluency2', density=True)
ax.hist(df25_fair['human_score'], bins=30, alpha=0.6,
        color='gray', label='human', density=True)
ax.set_xlabel('Score')
ax.set_ylabel('Density')
ax.set_title('Score Distribution: fluency2 vs Human\n(WMT25)')
ax.legend()

# Residuals by pair
ax = axes[2]
pairs_ordered = df25_fair.groupby('pair')['disagreement'].mean().sort_values(
    ascending=False).index
residuals_by_pair = [
    df25_fair[df25_fair['pair']==lp]['disagreement'].values
    for lp in pairs_ordered
]
ax.boxplot(residuals_by_pair,
           labels=[p.replace('EN→','') for p in pairs_ordered],
           vert=True)
ax.set_xlabel('Language Pair')
ax.set_ylabel('|fluency2_z - human_z|')
ax.set_title('Error Distribution by Language Pair')
plt.xticks(rotation=45, ha='right')

plt.suptitle('Calibration Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/fig_calibration.png', dpi=150)
plt.show()


### 9. Summary


In [ ]:
print("=" * 60)
print("SUMMARY: fluency2 Error Analysis")
print("=" * 60)

# 1. Overall
print(f"\n1. OVERALL PERFORMANCE (fair subset)")
print(f"   WMT25 ESA: n={len(df25_fair)}")
rho25, _ = spearmanr(df25_fair['fluency2'], df25_fair['human_score'])
print(f"   Spearman: {rho25:.3f}")

# 2. Worst pairs
print(f"\n2. WORST LANGUAGE PAIRS FOR fluency2")
worst_pairs = df25_fair.groupby('pair').apply(
    lambda g: spearmanr(g['fluency2'], g['human_score'])[0]
).sort_values().head(3)
print(worst_pairs.round(3).to_string())

# 3. Best pairs
print(f"\n3. BEST LANGUAGE PAIRS FOR fluency2")
best_pairs = df25_fair.groupby('pair').apply(
    lambda g: spearmanr(g['fluency2'], g['human_score'])[0]
).sort_values(ascending=False).head(3)
print(best_pairs.round(3).to_string())

# 4. Length
print(f"\n4. WORST LENGTH BUCKET")
worst_len = df_len[df_len['metric']=='fluency2'].nsmallest(1, 'spearman')
print(worst_len[['bucket','spearman','n']].to_string(index=False))

# 5. Overestimate rate
over_rate = df25_fair['f2_overestimates'].mean()
print(f"\n5. OVERESTIMATE RATE: {over_rate:.1%}")
print(f"   (fluency2 scores too HIGH vs human)")

# 6. low_confidence utility
if 'low_confidence' in df25_fair.columns:
    lc_mean = df25_fair.groupby('low_confidence')['disagreement'].mean()
    print(f"\n6. LOW_CONFIDENCE FLAG UTILITY")
    print(f"   low_conf=False: mean error={lc_mean.get(False, 'N/A'):.3f}")
    print(f"   low_conf=True:  mean error={lc_mean.get(True, 'N/A'):.3f}")
    print(f"   → Flag is {'USEFUL' if lc_mean.get(True, 0) > lc_mean.get(False, 0) else 'NOT USEFUL'}")

print("\n" + "=" * 60)


---
# Part 2. Failure analysis. What does fluency2 NOT understand?
Based on 1000 worst disagreements on WMT25 ESA and WMT22-24 MQM.

In [ ]:
def _resolve_fluency2_error_bundle() -> Path:
    candidates = [
        ROOT / 'data/fluency2_error_analysis_bundle',
        Path('/tmp/bundle/fluency2_error_analysis_bundle'),
    ]
    marker = 'wmt25_top_1000_fluency2_disagreements.csv'
    for d in candidates:
        if (d / marker).exists():
            return d
    raise FileNotFoundError(
        'fluency2_error_analysis_bundle not found. Tried:\n  '
        + '\n  '.join(str(c) for c in candidates)
        + '\nPlace or export the bundle there (see export_fluency2_error_analysis_bundle).'
    )


BUNDLE = _resolve_fluency2_error_bundle()

df_fail25 = pd.read_csv(
    BUNDLE / 'wmt25_top_1000_fluency2_disagreements.csv')
df_fail_mqm = pd.read_csv(
    BUNDLE / 'mqm_top_1000_fluency2_disagreements.csv')
df_lp_stats = pd.read_csv(
    BUNDLE / 'wmt25_per_language_pair_stats.csv')
df_quint25 = pd.read_csv(
    BUNDLE / 'wmt25_kendall_tau_by_human_quintile.csv')
df_quint_mqm = pd.read_csv(
    BUNDLE / 'mqm_kendall_tau_by_human_quintile.csv')
df_subdim_mat = pd.read_csv(
    BUNDLE / 'wmt25_subdimension_spearman_matrix.csv')
df_issue_dist = pd.read_csv(
    BUNDLE / 'wmt25_issue_type_distribution_by_pair.csv')
df_len_stats = pd.read_csv(
    BUNDLE / 'wmt25_length_bucket_stats.csv')

def _ensure_f2_overestimates(df: pd.DataFrame, human_col: str, label: str) -> pd.DataFrame:
    """Older bundles omitted `f2_overestimates` on MQM CSVs; fill when missing."""
    if 'f2_overestimates' in df.columns:
        return df
    if 'fluency2' not in df.columns or human_col not in df.columns:
        raise KeyError(
            f"{label}: need fluency2 and {human_col} to derive f2_overestimates; "
            f"got columns {list(df.columns)}"
        )
    out = df.copy()
    f2_z = (out['fluency2'] - out['fluency2'].mean()) / out['fluency2'].std(ddof=0)
    hz = (out[human_col] - out[human_col].mean()) / out[human_col].std(ddof=0)
    out['f2_overestimates'] = f2_z > hz
    print(
        f"WARNING [{label}]: CSV had no f2_overestimates; computed z-scores on this file only. "
        "Re-export with the benchmarks repo script algebras-ml/tools/export_fluency2_error_analysis_bundle.py for exact values."
    )
    return out


df_fail25 = _ensure_f2_overestimates(df_fail25, 'human_score', 'WMT25 failures')
df_fail_mqm = _ensure_f2_overestimates(df_fail_mqm, 'human_mqm_score', 'MQM failures')

df_fail25['error_direction'] = df_fail25['f2_overestimates'].map(
    {True: 'OVERESTIMATE', False: 'UNDERESTIMATE'})
df_fail_mqm['error_direction'] = df_fail_mqm['f2_overestimates'].map(
    {True: 'OVERESTIMATE', False: 'UNDERESTIMATE'})

print(f"WMT25 failures loaded: {len(df_fail25)}")
print(f"MQM failures loaded:   {len(df_fail_mqm)}")
print(f"Overall overestimate rate WMT25: "
      f"{df_fail25['f2_overestimates'].mean():.1%}")
print(f"Overall overestimate rate MQM:   "
      f"{df_fail_mqm['f2_overestimates'].mean():.1%}")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Issue type frequency colored by direction
ax = axes[0][0]
issue_stats = df_fail25.groupby('issue_type').agg(
    count=('issue_type', 'count'),
    overest_rate=('f2_overestimates', 'mean'),
    mean_disagree=('disagreement', 'mean')
).sort_values('count', ascending=True)
colors_bar = ['#D04040' if v > 0.5 else '#4080D0'
              for v in issue_stats['overest_rate']]
bars = ax.barh(issue_stats.index, issue_stats['count'],
               color=colors_bar, alpha=0.85)
ax.set_xlabel('Count in top-1000 failures')
ax.set_title('Issue Type Distribution\n'
             '(red=overestimate, blue=underestimate)')
for i, (idx, row) in enumerate(issue_stats.iterrows()):
    ax.text(row['count'] + 3, i,
            f"{row['overest_rate']:.0%} over | n={int(row['count'])}",
            va='center', fontsize=8)

# 2. Severity distribution
ax = axes[0][1]
sev_order = ['critical', 'major', 'minor', 'cosmetic']
sev_counts = df_fail25['severity'].value_counts().reindex(
    sev_order).fillna(0)
sev_colors = ['#8B0000', '#D04040', '#F4A460', '#90EE90']
ax.bar(sev_counts.index, sev_counts.values,
       color=sev_colors, alpha=0.85)
ax.set_ylabel('Count')
ax.set_title('Severity Distribution in Top-1000 Failures')
for i, (idx, v) in enumerate(sev_counts.items()):
    ax.text(i, v + 5, str(int(v)), ha='center', fontsize=10)

# 3. Overestimate rate by issue type
ax = axes[1][0]
overest_by_issue = df_fail25.groupby('issue_type')[
    'f2_overestimates'].mean().sort_values(ascending=False)
colors_bar2 = ['#D04040' if v > 0.5 else '#4080D0'
               for v in overest_by_issue.values]
ax.barh(overest_by_issue.index, overest_by_issue.values,
        color=colors_bar2, alpha=0.85)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.7,
           label='50% = random')
ax.set_xlabel('Fraction where fluency2 OVERESTIMATES')
ax.set_title('Overestimate Rate by Issue Type\n'
             '(>50% = fluency2 blindspot)')
ax.legend(fontsize=8)

# 4. Mean disagreement by issue type
ax = axes[1][1]
mean_disagree = df_fail25.groupby('issue_type')[
    'disagreement'].mean().sort_values(ascending=True)
ax.barh(mean_disagree.index, mean_disagree.values,
        color=COLORS['fluency2'], alpha=0.8)
ax.set_xlabel('Mean |fluency2_z - human_z|')
ax.set_title('Mean Error Magnitude by Issue Type')

plt.suptitle(
    'fluency2 Failure Taxonomy — WMT25 Top-1000 Disagreements',
    fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/fig_taxonomy.png', dpi=150)
plt.show()


In [ ]:
print("=" * 60)
print("FAILURE TYPE 1: REGISTER AND DIALECT BLINDNESS")
print("=" * 60)
print("Pattern:   Translation in wrong dialect but reads fluent")
print("Direction: OVERESTIMATE (73.8% of register_mismatch cases)")
print("=" * 60)

reg_df = df_fail25[
    df_fail25['issue_type'] == 'register_mismatch'].copy()
reg_df = reg_df.sort_values('disagreement', ascending=False)

print(f"\nTotal register mismatch failures: {len(reg_df)}")
print(f"By language pair:")
print(reg_df.groupby('pair').agg(
    count=('pair','count'),
    mean_disagree=('disagreement','mean'),
    overest_rate=('f2_overestimates','mean')
).sort_values('count', ascending=False).round(3).to_string())

print("\n=== TOP 10 EXAMPLES ===")
worst_reg = reg_df[reg_df['f2_overestimates']==True].head(10)
for i, (_, r) in enumerate(worst_reg.iterrows()):
    print(f"\n[{i+1}] {r['pair']} | "
          f"Human: {r['human_score']:.0f} | "
          f"fluency2: {r['fluency2']:.2f}")
    print(f"     {str(r['main_issue'])[:160]}")
    print(f"     MT: {str(r['mt_text'])[:120]}")

print("\n=== FIX FOR fluency3_1 ===")
print("""Add as FIRST check in system prompt:
CRITICAL: Verify target language variety matches locale.
  ar_EG → Egyptian Arabic (عامية), NOT MSA
  pt_BR → Brazilian Portuguese, NOT European
  zh_CN → Simplified Chinese, NOT Traditional
Wrong dialect/register = score 0-2 regardless of fluency.""")


In [ ]:
print("=" * 60)
print("FAILURE TYPE 2: FLUENCY BIAS")
print("=" * 60)
print("Pattern:   No error found → assumes high quality")
print("Direction: OVERESTIMATE (98.6% of 'none' issue cases)")
print("=" * 60)

none_df = df_fail25[df_fail25['issue_type'] == 'none'].copy()
none_df = none_df.sort_values('disagreement', ascending=False)

print(f"\n'None issue' failures: {len(none_df)}")
print(f"Mean fluency2: {none_df['fluency2'].mean():.3f}")
print(f"Mean human:    {none_df['human_score'].mean():.1f}")

subdims = ['idiomatic','collocational','discourse','pragmatic','calque']
print("\nMean subdimension scores (model thinks it's fine):")
for s in subdims:
    if s in none_df.columns:
        v = none_df[none_df['f2_overestimates']==True][s].mean()
        print(f"  {s}: {v:.2f}")

print("\n=== TOP 10 EXAMPLES ===")
worst_none = none_df[none_df['f2_overestimates']==True].head(10)
for i, (_, r) in enumerate(worst_none.iterrows()):
    print(f"\n[{i+1}] {r['pair']} | "
          f"Human: {r['human_score']:.0f} | "
          f"fluency2: {r['fluency2']:.2f}")
    print(f"     {str(r['main_issue'])[:160]}")
    print(f"     SRC: {str(r['src_text'])[:100]}")
    print(f"     MT:  {str(r['mt_text'])[:100]}")

print("\n=== FIX FOR fluency3_1 ===")
print("""Add skepticism probe:
Before scoring above 7, explicitly verify:
1. Any meaning loss vs source?
2. Would a human WRITER produce this phrasing?
3. Any subtle tonal/cultural issues?
If uncertain → default to 6, not 8.""")


In [ ]:
print("=" * 60)
print("FAILURE TYPE 3: CALQUE OVER-PENALIZATION")
print("=" * 60)
print("Pattern:   fluency2 penalizes calques MORE than humans")
print("Direction: UNDERESTIMATE (71.1% of calque cases)")
print("Impact:    467/1000 — LARGEST single failure category")
print("=" * 60)

calque_df = df_fail25[df_fail25['issue_type'] == 'calque'].copy()
calque_df = calque_df.sort_values('disagreement', ascending=False)

print(f"\nCalque failures: {len(calque_df)}")
print(f"Underestimate rate: {1-calque_df['f2_overestimates'].mean():.1%}")
print(f"Mean disagreement: {calque_df['disagreement'].mean():.3f}")
if 'calque' in calque_df.columns:
    print(f"Mean calque subdim score: {calque_df['calque'].mean():.2f}")

print("\n=== TOP 10 UNDERESTIMATE EXAMPLES ===")
print("fluency2 scores LOW but human scores HIGH\n")
worst_calque = calque_df[
    calque_df['f2_overestimates']==False].head(10)
for i, (_, r) in enumerate(worst_calque.iterrows()):
    print(f"\n[{i+1}] {r['pair']} | "
          f"Human: {r['human_score']:.0f} | "
          f"fluency2: {r['fluency2']:.2f} | "
          f"calque dim: {r.get('calque','?'):.1f}")
    print(f"     {str(r['main_issue'])[:160]}")
    print(f"     MT: {str(r['mt_text'])[:100]}")

print("\n=== FIX FOR fluency3_1 ===")
print("""Recalibrate calque subdimension:
Penalize ONLY when unnatural to native speaker.
Do NOT penalize:
  - Common loanwords
  - Established borrowed structures
  - Technical terminology with no natural equivalent
Calibration: 5-6 mild calque, 3-4 obvious unnatural,
1-2 only when entire sentence is word-for-word.""")


In [ ]:
print("=" * 60)
print("FAILURE TYPE 4: FALSE FRIEND ERRORS")
print("=" * 60)
print("Pattern:   Correctly flags but sometimes misidentifies")
print("Direction: UNDERESTIMATE (77.2% — mostly correct direction)")
print("=" * 60)

ff_df = df_fail25[df_fail25['issue_type'] == 'false_friend'].copy()
ff_df = ff_df.sort_values('disagreement', ascending=False)

print(f"\nFalse friend failures: {len(ff_df)}")
print(f"Underestimate rate: {1-ff_df['f2_overestimates'].mean():.1%}")

print("\n=== ALL FALSE FRIEND CASES ===\n")
for i, (_, r) in enumerate(ff_df.iterrows()):
    d = "OVER" if r['f2_overestimates'] else "UNDER"
    print(f"[{i+1}] {r['pair']} | Human: {r['human_score']:.0f} | "
          f"fluency2: {r['fluency2']:.2f} | {d}")
    print(f"     {str(r['main_issue'])[:160]}")
    print()


In [ ]:
print("=" * 60)
print("FAILURE TYPE 5: LOW-RESOURCE LANGUAGE BLINDNESS")
print("=" * 60)
print("Pattern:   Cannot judge quality for languages with")
print("           minimal LLM training data")
print("Languages: EN→mas_KE (Maasai), EN→bho_IN (Bhojpuri)")
print("=" * 60)

f2_lp = df_lp_stats[df_lp_stats['metric']=='fluency2'].copy()
f2_lp = f2_lp.sort_values('spearman', ascending=False)

print("\n=== fluency2 PERFORMANCE BY LANGUAGE PAIR ===")
print(f2_lp[['pair','spearman','pa','morphology',
              'script','family']].to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
colors_lp = ['#D04040' if v < 0.35 else
             '#F4A460' if v < 0.5 else '#2D4A8A'
             for v in f2_lp['spearman']]
ax.barh(f2_lp['pair'], f2_lp['spearman'],
        color=colors_lp, alpha=0.85)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Spearman ρ')
ax.set_title('fluency2 Correlation by Language Pair\n'
             'Red=poor, Yellow=mediocre, Blue=good')

ax = axes[1]
morph_perf = f2_lp.groupby('morphology')['spearman'].agg(
    ['mean','std','count'])
morph_perf = morph_perf.sort_values('mean', ascending=False)
ax.bar(morph_perf.index, morph_perf['mean'],
       yerr=morph_perf['std'], capsize=5,
       color=COLORS['fluency2'], alpha=0.8)
ax.set_xlabel('Morphological Type')
ax.set_ylabel('Mean Spearman ρ')
ax.set_title('fluency2 by Morphological Type')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')

plt.suptitle('Language Pair Analysis: Where Does fluency2 Fail?',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/fig_lp_analysis.png', dpi=150)
plt.show()

# Maasai special case
print("\n=== EN→mas_KE SPECIAL CASE ===")
mas = df_fail25[df_fail25['pair']=='EN→mas_KE']
print(f"Maasai in top-1000: {len(mas)}")
print(f"Overestimate rate: {mas['f2_overestimates'].mean():.1%} (≈random)")
print("Root cause: Gemini 3 Flash insufficient training data in Maasai")
print("Fix: Flag as LOW_CONFIDENCE by default; ensemble with supervised metric")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metric_colors_map = {
    'fluency2': '#2D4A8A', 'gemba_score': '#8A4A2D',
    'gemba_mqm': '#8A4A2D', 'gemba_da': '#C4763A',
    'metricx_score': '#2D8A5A', 'xcomet_score': '#5A2D8A',
    'cometKiwi_score': '#8A7A2D', 'chrf_pp': '#8A2D5A',
}

for ax_i, (df_q, title, metrics) in enumerate([
    (df_quint25,
     'WMT25 ESA',
     ['fluency2','gemba_score','metricx_score',
      'xcomet_score','cometKiwi_score','chrf_pp']),
    (df_quint_mqm,
     'WMT22 MQM',
     ['fluency2','gemba_mqm','gemba_da','chrf_pp']),
]):
    ax = axes[ax_i]
    for metric in metrics:
        sub = df_q[df_q['metric'] == metric]
        if sub.empty:
            continue
        color = metric_colors_map.get(metric, '#888')
        lw = 3 if metric == 'fluency2' else 1.5
        ls = '-' if metric == 'fluency2' else '--'
        label = (metric.replace('_score','')
                        .replace('_pp','')
                        .replace('_mqm',' MQM')
                        .replace('_da',' DA'))
        ax.plot(sub['quintile'].astype(str), sub['tau'],
                marker='o', color=color,
                linewidth=lw, linestyle=ls, label=label)

    ax.axhline(0, color='red', linestyle=':', alpha=0.7,
               label='τ=0 (random)')
    ax.set_xlabel('Human Score Quintile (Q1=worst)')
    ax.set_ylabel('Kendall τ')
    ax.set_title(f'{title}: Kendall τ by Quality Quintile')
    ax.legend(fontsize=7, loc='lower right')
    ax.set_ylim(-0.3, 0.75)

plt.suptitle(
    'Quality Quintile Analysis\n'
    'fluency2 is NEGATIVE on Q1 (worst translations) — '
    'worse than random',
    fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/fig_quintile_final.png', dpi=150)
plt.show()

print("=== KEY INSIGHT: Q1 COLLAPSE ===")
q1_f2 = df_quint25[
    (df_quint25['metric']=='fluency2') &
    (df_quint25['quintile'].str.contains('Q1'))
]['tau'].values[0]
print(f"fluency2 Kendall τ on worst translations: {q1_f2:.3f}")
print("Interpretation: on very bad translations, fluency2 is")
print("WORSE THAN RANDOM at ranking them correctly.")
print("Fix: Add critical error detection BEFORE fluency scoring.")


In [ ]:
print("=== SUBDIMENSION CORRELATION BY LANGUAGE PAIR ===")

# Sort EVERY column "by color" (palette order)
subdim_cols_all = ['calque','collocational','discourse','idiomatic','pragmatic']
subdim_cols_all = [c for c in subdim_cols_all if c in df_subdim_mat.columns]
ordered = [c for c in SUBDIM_ORDER if c in subdim_cols_all]
remaining = [c for c in subdim_cols_all if c not in ordered]
subdim_cols = ordered + remaining

subdim_data = df_subdim_mat.set_index('pair')[subdim_cols]
print(subdim_data.round(3).to_string())

print("\n=== SUBDIMENSION STATISTICS ACROSS ALL PAIRS ===")
print(subdim_data.agg(['mean','std','min','max']).round(3).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
if SEABORN_AVAILABLE:
    sns.heatmap(subdim_data, annot=True, fmt='.2f',
                cmap='RdYlGn', center=0.4, ax=ax,
                linewidths=0.5, vmin=0.1, vmax=0.85)
    # Color tick labels by subdim palette
    for tick in ax.get_xticklabels():
        name = tick.get_text()
        tick.set_color(SUBDIM_COLORS.get(name, 'black'))
else:
    im = ax.imshow(subdim_data.values, aspect='auto', cmap='RdYlGn', vmin=0.1, vmax=0.85)
    ax.set_xticks(range(subdim_data.shape[1]))
    ax.set_xticklabels(subdim_data.columns, rotation=20, ha='right')
    for tick in ax.get_xticklabels():
        name = tick.get_text()
        tick.set_color(SUBDIM_COLORS.get(name, 'black'))
    ax.set_yticks(range(subdim_data.shape[0]))
    ax.set_yticklabels(subdim_data.index)
    for i in range(subdim_data.shape[0]):
        for j in range(subdim_data.shape[1]):
            v = subdim_data.values[i, j]
            if pd.notna(v):
                ax.text(j, i, f"{v:.2f}", ha='center', va='center', fontsize=7)
    fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)

ax.set_title('Subdimension Spearman ρ by Language Pair\n'
             '(green=good, red=poor)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20)

ax = axes[1]
subdim_std = subdim_data.std().sort_values(ascending=False)
ax.bar(subdim_std.index, subdim_std.values,
       color=COLORS['fluency2'], alpha=0.8)
ax.set_ylabel('Std across language pairs')
ax.set_title('Subdimension Stability\n'
             '(higher = less consistent across languages)')

plt.suptitle('Subdimension Reliability',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/fig_subdim_reliability.png', dpi=150)
plt.show()

# Scores in overestimate vs underestimate
subdims_avail = [s for s in subdim_cols if s in df_fail25.columns]
if subdims_avail:
    print("\n=== SUBDIM SCORES: OVERESTIMATE vs UNDERESTIMATE ===")
    comp = df_fail25.groupby(
        'f2_overestimates')[subdims_avail].mean().round(3)
    comp.index = ['UNDERESTIMATE', 'OVERESTIMATE']
    print(comp.to_string())
    print("\nHigh scores in OVERESTIMATE row = model misses that issue")


## Summary: Priority Fixes for fluency3_1

| # | Failure Type | Failures | Direction | Fix |
|---|---|---|---|---|
| 1 | Calque over-penalization | 467/1000 | UNDERESTIMATE | Recalibrate calque scoring |
| 2 | Register/dialect blindness | 195/1000 | OVERESTIMATE | Add dialect check as first step |
| 3 | Fluency bias (no error found) | 69/1000 | OVERESTIMATE | Add skepticism probe |
| 4 | Q1 collapse (worst translations) | cross-cutting | mixed | Add critical error detection |
| 5 | Low-resource blindness | Maasai/Bhojpuri | random | Flag + ensemble |
| 6 | False friend misidentification | 57/1000 | UNDERESTIMATE | Improve span localization |

In [ ]:
print("=" * 65)
print("ACTIONABLE CHANGES FOR fluency3_1 SYSTEM PROMPT")
print("=" * 65)
print("""
CHANGE 1 — Add BEFORE all other scoring (register/dialect):
─────────────────────────────────────────────────────────────
STEP 0 - DIALECT CHECK (mandatory):
Verify the target language variety matches the locale:
  ar_EG → Egyptian Arabic colloquial, NOT Modern Standard Arabic
  pt_BR → Brazilian Portuguese, NOT European Portuguese
  zh_CN → Simplified Chinese, NOT Traditional Chinese
If the variety is WRONG → all scores capped at 2/10.

CHANGE 2 — Recalibrate calque dimension:
─────────────────────────────────────────────────────────────
calque: Penalize ONLY when a native speaker would notice
the text sounds translated. Do NOT penalize:
  - Common loanwords and borrowed terms
  - Technical terminology with no natural equivalent
  - Structures that are natural in the target language
Scoring: 7-8 = slight mirror structure, acceptable
         5-6 = noticeable but not jarring
         3-4 = clearly unnatural source-induced structure
         1-2 = entire text reads as word-for-word translation

CHANGE 3 — Add anti-bias skepticism check:
─────────────────────────────────────────────────────────────
Before assigning ANY score above 7, verify:
1. Is there ANY meaning loss compared to the source?
2. Would a HUMAN WRITER produce this exact phrasing?
3. Are there subtle tonal/cultural issues?
If uncertain about any of the above → score 5-6, not 7-8.

CHANGE 4 — Add critical error detection at top:
─────────────────────────────────────────────────────────────
STEP -1 (before all scoring):
Check for critical failures:
  - Translation is in a completely wrong language
  - Complete mistranslation (opposite meaning)
  - More than 30% of content missing
If any → all scores = 1-2, stop evaluation.
""")


### Add-on: Export failure docs + quick examples

This section:
- Writes all detected failures into `output/fluency2_failure_docs/` (Markdown + CSV)
- Prints **5 examples per issue type** (WMT25 top-1000)
- Provides a small legend table explaining what each `issue_type` means

In [ ]:
from datetime import datetime

OUT = ROOT / 'output' / 'fluency2_failure_docs'
OUT.mkdir(parents=True, exist_ok=True)

def _safe_cols(df, cols):
    return [c for c in cols if c in df.columns]


def _md_table(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return "```\n" + df.to_string(index=False) + "\n```\n"


def _ensure_failure_dfs():
    """Load top-failure tables if this cell runs before the Part 2 bundle cell."""
    g = globals()
    if (
        'df_fail25' in g
        and isinstance(g.get('df_fail25'), pd.DataFrame)
        and 'df_fail_mqm' in g
        and isinstance(g.get('df_fail_mqm'), pd.DataFrame)
    ):
        return
    bundle = None
    b = g.get('BUNDLE')
    marker = Path('wmt25_top_1000_fluency2_disagreements.csv')
    if isinstance(b, Path) and (b / marker.name).exists():
        bundle = b
    else:
        for d in (
            Path('/tmp/bundle/fluency2_error_analysis_bundle'),
            ROOT / 'output' / 'fluency2_error_analysis_bundle',
        ):
            if (d / marker).exists():
                bundle = d
                break
    if bundle is None:
        raise FileNotFoundError(
            'df_fail25 / df_fail_mqm not loaded and bundle CSV not found. '
            'Run the Part 2 cell that sets BUNDLE and loads the tables, '
            'or put fluency2_error_analysis_bundle under data/ or /tmp/bundle/.'
        )
    g['df_fail25'] = pd.read_csv(bundle / 'wmt25_top_1000_fluency2_disagreements.csv')
    g['df_fail_mqm'] = pd.read_csv(bundle / 'mqm_top_1000_fluency2_disagreements.csv')
    if 'error_direction' not in g['df_fail25'].columns:
        g['df_fail25']['error_direction'] = g['df_fail25']['f2_overestimates'].map(
            {True: 'OVERESTIMATE', False: 'UNDERESTIMATE'})
    if 'error_direction' not in g['df_fail_mqm'].columns:
        g['df_fail_mqm']['error_direction'] = g['df_fail_mqm']['f2_overestimates'].map(
            {True: 'OVERESTIMATE', False: 'UNDERESTIMATE'})


_ensure_failure_dfs()
_g = globals()
df_fail25 = _g['df_fail25']
df_fail_mqm = _g['df_fail_mqm']

# --- Legend: what does each issue type mean? ---
ISSUE_TYPE_LEGEND = {
    'calque': {
        'task_type': 'lexico-syntactic naturalness',
        'meaning': 'Source-induced literal phrasing / translated-sounding structure.',
    },
    'register_mismatch': {
        'task_type': 'sociolinguistic / locale compliance',
        'meaning': 'Wrong dialect/variety/register for the locale (e.g., ar_EG vs MSA).',
    },
    'false_friend': {
        'task_type': 'lexical semantics',
        'meaning': 'Deceptively similar word/phrase leads to incorrect meaning choice.',
    },
    'none': {
        'task_type': 'missing-error detection / meaning fidelity',
        'meaning': 'No explicit issue flagged, but human score disagrees strongly (often meaning loss).',
    },
}

# Extend legend with any other labels present
_issue_vals = (
    df_fail25['issue_type'].dropna().astype(str).unique()
    if 'issue_type' in df_fail25.columns
    else []
)
for it in sorted(set(_issue_vals)):
    ISSUE_TYPE_LEGEND.setdefault(it, {
        'task_type': 'other/unspecified',
        'meaning': 'Not documented in legend yet (auto-added).',
    })

legend_df = (
    pd.DataFrame.from_dict(ISSUE_TYPE_LEGEND, orient='index')
      .reset_index()
      .rename(columns={'index': 'issue_type'})
      [['issue_type', 'task_type', 'meaning']]
)

legend_path = OUT / 'issue_type_legend.csv'
legend_df.to_csv(legend_path, index=False)
print(f'Wrote legend: {legend_path}')

# --- Export: all failures (WMT25 + MQM) ---
# These dataframes were loaded from data/fluency2_error_analysis_bundle (or /tmp/bundle/) in the previous cells.

fail25_path = OUT / 'wmt25_top_1000_failures.csv'
failmqm_path = OUT / 'mqm_top_1000_failures.csv'

df_fail25.to_csv(fail25_path, index=False)
df_fail_mqm.to_csv(failmqm_path, index=False)
print(f'Wrote: {fail25_path}')
print(f'Wrote: {failmqm_path}')

# --- Export: markdown report (human-readable) ---
md_path = OUT / 'fluency2_failure_report.md'
now = datetime.now().strftime('%Y-%m-%d %H:%M')

# Pick a compact set of columns to include in examples
example_cols = _safe_cols(df_fail25, [
    'pair', 'issue_type', 'severity', 'disagreement', 'f2_overestimates',
    'human_score', 'fluency2', 'main_issue', 'src_text', 'mt_text'
])

lines = []
lines.append(f"# fluency2 Failure Report\n")
lines.append(f"Generated: {now}\n")
lines.append(f"\n## Files\n")
lines.append(f"- WMT25 failures CSV: `{fail25_path}`\n")
lines.append(f"- MQM failures CSV: `{failmqm_path}`\n")
lines.append(f"- Issue legend CSV: `{legend_path}`\n")

# Summary stats
lines.append("\n## High-level stats\n")
lines.append(f"- WMT25 failures: {len(df_fail25)}\n")
lines.append(f"- MQM failures: {len(df_fail_mqm)}\n")
if 'f2_overestimates' in df_fail25.columns:
    lines.append(f"- WMT25 overestimate rate: {df_fail25['f2_overestimates'].mean():.1%}\n")

# Issue distribution
if 'issue_type' in df_fail25.columns:
    issue_counts = df_fail25['issue_type'].value_counts(dropna=False)
    lines.append("\n## Issue types (WMT25 top-1000)\n")
    lines.append(_md_table(issue_counts.to_frame('count')))
    lines.append("\n")

# Legend
lines.append("\n## Legend: issue_type → task type\n")
lines.append(_md_table(legend_df))
lines.append("\n")

# 5 examples per issue_type
if 'issue_type' in df_fail25.columns:
    lines.append("\n## 5 examples per issue_type (WMT25)\n")
    for it, grp in df_fail25.groupby('issue_type'):
        lines.append(f"\n### {it}\n")
        g2 = grp.sort_values('disagreement', ascending=False).head(5)
        # Make long text more readable
        gg = g2[example_cols].copy()
        for c in ['main_issue', 'src_text', 'mt_text']:
            if c in gg.columns:
                gg[c] = gg[c].astype(str).str.slice(0, 220)
        lines.append(_md_table(gg))
        lines.append("\n")

md_path.write_text("\n".join(lines), encoding='utf-8')
print(f'Wrote report: {md_path}')

# Also export the same 5-per-type examples as CSV for Claude
if 'issue_type' in df_fail25.columns:
    examples = (
        df_fail25.sort_values('disagreement', ascending=False)
                .groupby('issue_type', dropna=False)
                .head(5)
    )
    examples_path = OUT / 'wmt25_examples_5_per_issue_type.csv'
    examples[_safe_cols(examples, example_cols)].to_csv(examples_path, index=False)
    print(f'Wrote examples: {examples_path}')


In [ ]:
# Quick view in-notebook: show 5 examples per issue_type (WMT25)
if 'df_fail25' not in globals() or not isinstance(globals().get('df_fail25'), pd.DataFrame):
    _b = globals().get('BUNDLE')
    _bundle = None
    if isinstance(_b, Path) and (_b / 'wmt25_top_1000_fluency2_disagreements.csv').exists():
        _bundle = _b
    else:
        for _d in (ROOT / 'data/fluency2_error_analysis_bundle', Path('/tmp/bundle/fluency2_error_analysis_bundle')):
            if (_d / 'wmt25_top_1000_fluency2_disagreements.csv').exists():
                _bundle = _d
                break
    if _bundle is None:
        raise FileNotFoundError('df_fail25 not loaded; put bundle under data/fluency2_error_analysis_bundle or /tmp/bundle/ and re-run.')
    df_fail25 = pd.read_csv(_bundle / 'wmt25_top_1000_fluency2_disagreements.csv')

if 'issue_type' in df_fail25.columns:
    show_cols = [c for c in [
        'pair','issue_type','severity','disagreement',
        'human_score','fluency2','f2_overestimates','main_issue'
    ] if c in df_fail25.columns]

    print("=== 5 EXAMPLES PER ISSUE TYPE (sorted by disagreement) ===")
    for it, g in df_fail25.groupby('issue_type'):
        print("\n" + "-" * 80)
        print(f"ISSUE TYPE: {it} | n={len(g)}")
        display(g.sort_values('disagreement', ascending=False)[show_cols].head(5))
else:
    print("df_fail25 has no 'issue_type' column")


## Headline figures (MQM three-judge, `data/error_analysis`)

Validated outputs under `data/error_analysis/` are read-only here (no validation re-run, no edits to slice parquets). The next cell regenerates headline PNGs, caption markdown, the en-de / en-ru examples table, and the paper limitations snippet from those frozen artifacts.

In [ ]:
import subprocess
import sys

_script = ROOT / "tools/build_fluency2_error_analysis_headline_artifacts.py"
_sub = subprocess.run([sys.executable, str(_script)], cwd=str(ROOT), capture_output=True, text=True)
print(_sub.stdout or "")
if _sub.returncode != 0:
    raise RuntimeError(_sub.stderr or "headline build failed")

### Headline Figure 1: Variance collapse grid

Score histograms (0–100) vs human MQM rescaled to 0–100; saturation line at 98. Annotation text (n, pct≥98, std) is taken from `tables/variance_collapse.csv`.

![](data/error_analysis/figures/fig_headline_1_variance_collapse.png)

In [ ]:
from IPython.display import Markdown, display

_cap1 = ROOT / "data/error_analysis/figures/fig_headline_1_caption.md"
display(Markdown(_cap1.read_text(encoding="utf-8")))

### Headline Figure 2: Calibration vs variance trade-off

Scatter uses `std_raw_on_0_100` from `variance_collapse.csv` and `linreg_slope` from `bias_calibration.csv` (joined on mode × judge).

![](data/error_analysis/figures/fig_headline_2_calibration_variance.png)

In [ ]:
from IPython.display import Markdown, display

_cap2 = ROOT / "data/error_analysis/figures/fig_headline_2_caption.md"
display(Markdown(_cap2.read_text(encoding="utf-8")))

## Error examples (en-de & en-ru)

Curated from the frozen slice parquets under `data/error_analysis/`: for each slice, up to five **en-ru** and five **en-de** segments ranked by the slice-specific criterion (absolute residual for Slice A, worst human MQM for Slice B, pooled z for Slice C). Full `SRC` / `MT` / `REF` text is shown inside collapsible blocks; the same rows are exported to `tables/error_examples_en_de_en_ru.csv`.

In [ ]:
from IPython.display import HTML, display

_html_path = ROOT / "data/error_analysis/tables/error_examples_display.html"
display(HTML(_html_path.read_text(encoding="utf-8")))

In [ ]:
import json
from pathlib import Path

OUT = ROOT / "data/error_analysis"
expected_artifacts = [
    "figures/fig_headline_1_variance_collapse.png",
    "figures/fig_headline_1_caption.md",
    "figures/fig_headline_2_calibration_variance.png",
    "figures/fig_headline_2_caption.md",
    "tables/error_examples_en_de_en_ru.csv",
    "paper_limitations_snippet.md",
]
missing = [a for a in expected_artifacts if not (OUT / a).exists()]
assert not missing, f"missing outputs: {missing}"

_nb_path = ROOT / "fluency2_error_analysis.ipynb"
with open(_nb_path, encoding="utf-8") as f:
    nb = json.load(f)
cell_headers = [
    "".join(c.get("source", []))[:200] for c in nb["cells"] if c["cell_type"] == "markdown"
]
assert any("Headline Figure 1" in h for h in cell_headers), "Headline Figure 1 section missing"
assert any("Error examples" in h for h in cell_headers), "Error examples section missing"
print("[ok] notebook updated, all deliverables present")